# `RunnableParallel: RunnableSerializable[Input, dict[str, Any]]`

`RunnableParallel` runs multiple `Runnable` objects concurrently with the same input and returns their outputs in a dictionary.

Each dictionary key identifies one parallel step and its corresponding result.

## Type Parameter

```python
Input # Input type passed to every parallel step
```

## Field

```python
steps__: Mapping[str, Runnable[Input, Any]] # Named Runnables executed in parallel
```

## Constructor

```python
RunnableParallel(
    steps__: Mapping[
        str,
        Runnable[Input, Any]
        | Callable[[Input], Any]
        | Mapping[str, Runnable[Input, Any] | Callable[[Input], Any]],
    ] | None = None, # Optional mapping of named parallel steps
    **kwargs: Runnable[Input, Any]
    | Callable[[Input], Any]
    | Mapping[str, Runnable[Input, Any] | Callable[[Input], Any]], # Additional named parallel steps
) -> None # Initialize the RunnableParallel
```

Callables and nested mappings supplied to the constructor are automatically converted into `Runnable` objects.

## Input and Output

```python
Input # Same input supplied to every step
dict[str, Any] # Dictionary containing the output of each named step
```

## Overridden Properties and Methods

### `is_lc_serializable`

Returns `True`, indicating that `RunnableParallel` supports LangChain serialization.

### `get_lc_namespace`

Returns the LangChain serialization namespace for Runnable objects.

### `get_name`

Returns the custom name when provided.

Otherwise, it generates a name containing the keys of the parallel steps.

### `InputType`

Returns the input type of the first step that exposes one.

Returns `Any` when no step provides an inferable input type.

### `get_input_schema`

Combines compatible object-based input schemas from the parallel steps into one input schema.

Falls back to the normal Runnable input schema when the step schemas cannot be combined.

### `get_output_schema`

Returns an object schema in which every step name becomes a required field with that step's output type.

### `config_specs`

Returns the unique configurable-field specifications collected from all parallel steps.

### `get_graph`

Returns one execution graph containing all parallel branches between shared input and output nodes.

### `__repr__`

Returns a dictionary-like representation of the named parallel steps.

### `invoke`

Synchronously executes all steps concurrently with the same input.

Returns a dictionary containing each step's result under its configured key.

### `ainvoke`

Asynchronously executes all steps concurrently with the same input.

Returns a dictionary containing each step's result under its configured key.

### `transform`

Processes a synchronous input iterator through every step in parallel.

Yields dictionaries containing chunks from individual steps as those chunks become available.

### `stream`

Synchronously streams output chunks from the parallel steps.

Each yielded dictionary normally contains the key and latest chunk of one completed branch.

### `atransform`

Processes an asynchronous input iterator through every step concurrently.

Asynchronously yields dictionaries containing chunks as individual steps produce them.

### `astream`

Asynchronously streams output chunks from the parallel steps.

Each yielded dictionary normally contains the key and latest chunk of one branch.

## Alias

```python
RunnableMap = RunnableParallel # Alternative name for RunnableParallel
```

## Behaviour

- Every step receives the same input.
- Step outputs are stored under their configured keys.
- Synchronous execution uses parallel worker execution.
- Asynchronous execution runs step coroutines concurrently.
- Streaming emits chunks as branches produce them.
- Step order does not control completion order.

In [ ]:
from langchain_core.runnables import RunnableLambda, RunnableParallel # Import required Runnable classes

def square(number: int) -> int: # Define a function to calculate the square
    return number ** 2 # Return the squared value

def cube(number: int) -> int: # Define a function to calculate the cube
    return number ** 3 # Return the cubed value

parallel_runnable = RunnableParallel( # Create multiple Runnables that execute in parallel
    square=RunnableLambda(square), # Create the Runnable for calculating the square
    cube=RunnableLambda(cube), # Create the Runnable for calculating the cube
)

result = parallel_runnable.invoke(4) # Pass the same input to both Runnables

print(result) # Display all results as a dictionary